# Linear regression model

## Steps
1. Get the data
2. Explore and clean the data
3. separate features and target
4. Split the data into training and testing set 
5. Preprocess the data
   - handle missing value
   - encode categorical data
   - standardize numeric features
 
7. train the model
8. make prediction
9. evaluate your model

In [1]:
# import the needed libraries
import pandas as pd
from sklearn.preprocessing import OneHotEncoder,StandardScaler
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

In [2]:
fish_data =pd.read_csv("/kaggle/input/datasets/salman1127/fish-market-dataset/Fishers maket.csv")

In [3]:
fish_data

,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340
...,...,...,...,...,...,...,...
154,Smelt,12.2,11.5,12.2,13.4,2.0904,1.3936
155,Smelt,13.4,11.7,12.4,13.5,2.4300,1.2690
156,Smelt,12.2,12.1,13.0,13.8,2.2770,1.2558
157,Smelt,19.7,13.2,14.3,15.2,2.8728,2.0672


In [4]:
fish_data.Species.nunique()

7

In [5]:
fish_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Species  159 non-null    object 
 1   Weight   159 non-null    float64
 2   Length1  159 non-null    float64
 3   Length2  159 non-null    float64
 4   Length3  159 non-null    float64
 5   Height   159 non-null    float64
 6   Width    159 non-null    float64
dtypes: float64(6), object(1)
memory usage: 8.8+ KB


In [6]:
fish_data.Weight.corr(fish_data.Height)

np.float64(0.7243453291993318)

In [7]:
fish_data.Weight.corr(fish_data.Width)

np.float64(0.8865066052433445)

In [8]:
fish_data.Weight.corr(fish_data.Length1)

np.float64(0.915711716031204)

In [9]:
fish_data.Weight.corr(fish_data.Length2)

np.float64(0.9186177013642217)

In [10]:
fish_data.Weight.corr(fish_data.Length3)

np.float64(0.9230435593620121)

In [11]:
#encode categorical data : species
encoder =OneHotEncoder(sparse_output=False)
encoded =encoder.fit_transform(fish_data.Species.to_frame())
encoded_df =pd.DataFrame(encoded,columns=encoder.get_feature_names_out(fish_data.Species.to_frame().columns))

In [12]:
encoded_df

,Species_Bream,Species_Parkki,Species_Perch,Species_Pike,Species_Roach,Species_Smelt,Species_Whitefish
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
154,0.0,0.0,0.0,0.0,0.0,1.0,0.0
155,0.0,0.0,0.0,0.0,0.0,1.0,0.0
156,0.0,0.0,0.0,0.0,0.0,1.0,0.0
157,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [13]:
encoded_df.columns

Index(['Species_Bream', 'Species_Parkki', 'Species_Perch', 'Species_Pike',
       'Species_Roach', 'Species_Smelt', 'Species_Whitefish'],
      dtype='object')

In [14]:
new_fish_data =pd.concat([fish_data,encoded_df],axis=1)

In [15]:
new_fish_data

,Species,Weight,Length1,Length2,Length3,Height,Width,Species_Bream,Species_Parkki,Species_Perch,Species_Pike,Species_Roach,Species_Smelt,Species_Whitefish
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,Smelt,12.2,11.5,12.2,13.4,2.0904,1.3936,0.0,0.0,0.0,0.0,0.0,1.0,0.0
155,Smelt,13.4,11.7,12.4,13.5,2.4300,1.2690,0.0,0.0,0.0,0.0,0.0,1.0,0.0
156,Smelt,12.2,12.1,13.0,13.8,2.2770,1.2558,0.0,0.0,0.0,0.0,0.0,1.0,0.0
157,Smelt,19.7,13.2,14.3,15.2,2.8728,2.0672,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [16]:
new_fish_data.columns

Index(['Species', 'Weight', 'Length1', 'Length2', 'Length3', 'Height', 'Width',
       'Species_Bream', 'Species_Parkki', 'Species_Perch', 'Species_Pike',
       'Species_Roach', 'Species_Smelt', 'Species_Whitefish'],
      dtype='object')

In [17]:
new_fish_data.select_dtypes(include="number").corr()

,Weight,Length1,Length2,Length3,Height,Width,Species_Bream,Species_Parkki,Species_Perch,Species_Pike,Species_Roach,Species_Smelt,Species_Whitefish
Weight,1.000000,0.915712,0.918618,0.923044,0.724345,0.886507,0.326795,-0.186034,-0.033240,0.310641,-0.261784,-0.337109,0.073625
Length1,0.915712,1.000000,0.999517,0.992031,0.625378,0.867050,0.216380,-0.205732,-0.037845,0.563514,-0.213250,-0.467420,0.050731
Length2,0.918618,0.999517,1.000000,0.994103,0.640441,0.873547,0.233391,-0.205957,-0.036090,0.552780,-0.218048,-0.479775,0.053777
Length3,0.923044,0.992031,0.994103,1.000000,0.703409,0.878520,0.327170,-0.198718,-0.105478,0.522894,-0.205072,-0.488397,0.052864
Height,0.724345,0.625378,0.640441,0.703409,1.000000,0.792881,0.772443,-0.000547,-0.191405,-0.101810,-0.202076,-0.491731,0.048951
Width,0.886507,0.867050,0.873547,0.878520,0.792881,1.000000,0.319347,-0.194147,0.144021,0.137722,-0.171465,-0.569018,0.124388
Species_Bream,0.326795,0.216380,0.233391,0.327170,0.772443,0.319347,1.000000,-0.144840,-0.391741,-0.183825,-0.201526,-0.165083,-0.105209
Species_Parkki,-0.186034,-0.205732,-0.205957,-0.198718,-0.000547,-0.194147,-0.144840,1.000000,-0.201021,-0.094329,-0.103413,-0.084712,-0.053988
Species_Perch,-0.033240,-0.037845,-0.036090,-0.105478,-0.191405,0.144021,-0.391741,-0.201021,1.000000,-0.255127,-0.279694,-0.229116,-0.146018
Species_Pike,0.310641,0.563514,0.552780,0.522894,-0.101810,0.137722,-0.183825,-0.094329,-0.255127,1.000000,-0.131247,-0.107513,-0.068519


In [18]:
fig=px.scatter(new_fish_data,x="Length3",y="Weight",color="Species")
fig.update_traces(marker_size=5)
fig.show()

In [19]:
#separate target and features
x=fish_data.drop(columns="Weight")
y=fish_data.Weight

In [20]:
#split training and validation data
train_x,val_x,train_y,val_y=train_test_split(x,y,train_size=0.8,test_size=0.2,random_state=0)

In [21]:
numeric_cols =train_x.select_dtypes(include ="number").columns
categorical_cols =train_x.select_dtypes(include ="object").columns

In [22]:
#preprocessor for numeric cols
numerical_transformer =Pipeline(steps =[
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler())
])

#categorical col preprocessor
categorical_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ('onehot',OneHotEncoder(handle_unknown="ignore"))
])

#Bundle Together numeric and categorical preprocessing
preprocessor =ColumnTransformer(transformers=[('num',numerical_transformer,numeric_cols),
                                              ('cat',categorical_transformer,categorical_cols)])

In [23]:
#define the model
model = LinearRegression()

In [24]:
#bundle together preprocessing and modelling in pipeline
my_pipeline =Pipeline(steps =[
    ("preprocessor",preprocessor),
    ("model",model)
])

#train the model
my_pipeline.fit(train_x,train_y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['Length1', 'Length2', 'Length3', 'Height', 'Width'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Species'], dtype='object'))])),
                ('model', LinearRegression())])

In [25]:
#make prediction
preds=my_pipeline.predict(val_x)

In [26]:
mean_absolute_error(val_y,preds)

88.69881474243425

In [27]:
fish_data["Weight"].describe()

count     159.000000
mean      398.326415
std       357.978317
min         0.000000
25%       120.000000
50%       273.000000
75%       650.000000
max      1650.000000
Name: Weight, dtype: float64